# T01 · Algèbre linéaire — ADIF72 Mathematics for Data Scientists

**Objectifs du cours :**
- Manipuler les systèmes linéaires, matrices et transformations
- Comprendre rang, bases, sous-espaces (image / noyau)
- Utiliser les valeurs propres et la SVD pour analyser des données

**Pourquoi c'est fondamental en data science :**
| Besoin data science | Outil algébrique |
|---|---|
| Régression linéaire | Résolution de `Xθ = y` / moindres carrés |
| PCA (réduction de dimension) | SVD / valeurs propres de la matrice de covariance |
| Réseaux de neurones | Propagation = chaînes de produits matriciels |
| RecSys (recommandation) | Facteurs matriciels (SVD) |

**Pré-requis :** arithmatique matricielle de base. Si ce n'est pas le cas, relire
Deisenroth ch. 2 ou Lay ch. 1-3.

> 💡 *Conseil étude* : exécuter chaque cellule et **modifier les valeurs** pour
> voir ce qui change. L'algèbre linéaire se comprend en manipulant.


## 📌 Bloc 0 — Environnement & imports
Installez le stack si besoin (déjà pré-installé sur Colab).
Chaque notebook contient des sorties pré-calculées : vous pouvez les relire
sans rien exécuter, ou « Run All » pour reproduire.

In [ ]:
!pip install -q numpy scipy pandas matplotlib
import numpy as np
np.set_printoptions(precision=3, suppress=True, linewidth=100)
print("numpy", np.__version__)

## 1 · Opérations matricielles, systèmes & transformations linéaires
### Ce qu'on apprend
- **Produit matriciel** `A @ B` : compose deux transformations linéaires.
  `(AB)_{ij} = Σ_k A_{ik} B_{kj}`. C'est **l'opération centrale** de tout le ML.
- **Transpose** `A.T` : échange lignes/colonnes ; apparaît dans les moindres
  carrés (`XᵀX`).
- **Déterminant** `det(A)` : volume ( signé ) transformé ; si `det=0` la matrice
  est **singulière** (non inversible, rang strictement inférieur à n).
- **Inversée** `A⁻¹` : satisfait `AA⁻¹ = I`. N'existe que si `A` est carrée
  ET de rang plein.

### Piège classique
`A @ B ≠ B @ A` en général (le produit matriciel **n'est pas commutatif**).
Vérifiez toujours avec un petit code.

In [ ]:
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])
print("A @ B =\n", A @ B)
print("B @ A =\n", B @ A, " -> différent (non commutatif)")
print("det(A) =", np.linalg.det(A))
print("inv(A) =\n", np.linalg.inv(A))
print("A @ inv(A) ≈ Identité?\n", A @ np.linalg.inv(A))

### Systèmes linéaires  `Ax = b`

### Ce qu'on apprend
Un système de `n` équations à `n` inconnues correspond à une matrice `A`.
- Si `det(A) ≠ 0` : **solution unique** `x = A⁻¹b` (ou mieux : `np.linalg.solve`).
- Si `det(A) = 0` : **0 solution** (incompatible) ou **∞ solutions** (syst. dépendant).

### En data science
La régression par moindres carrés se résout comme un système :
le **système normal** `XᵀX θ = Xᵀy`. Avec une matrice de features mal conditionnée,
`XᵀX` devient singulière → il faut régulariser (Ridge, Lasso) — on y revient au T03.

In [ ]:
# 2x + 3y - z = 5 ;  x - y + 2z = 6 ;  3x + 2y + 2z = 9
A_sys = np.array([[2, 3, -1], [1, -1, 2], [3, 2, 2]])
b_sys = np.array([5, 6, 9])
x = np.linalg.solve(A_sys, b_sys)
print("Solution x =", x)
print("Vérif A@x =", A_sys @ x, "(= b attendu)")

# Cas singulier : 3e ligne = 1re + 2e
A_sing = np.array([[1, 1, 0], [1, -1, 0], [2, 0, 0]])
print("\nMatrice singulière det = ", np.linalg.det(A_sing),
      " rang =", np.linalg.matrix_rank(A_sing))

### Transformations linéaires (rotation)

Une rotation d'angle `θ` dans le plan :
```
R = [ cosθ  -sinθ ]
    [ sinθ   cosθ ]
```
Propriétés : `det(R)=1`, `R Rᵀ = I` (orthogonale), et la norme est conservée :
`‖Rv‖ = ‖v‖`.
En data science, les rotations apparaissent dans PCA (rotation des axes),
les CNN (invariance rotationnelle approximative), et les embeddings.

In [ ]:
theta = np.pi / 3   # 60°
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
v = np.array([1.0, 0.0])
print("R =\n", R)
print("Rotation de (1,0) sur 60° ->", R @ v)
print("Norme conservée :", np.isclose(np.linalg.norm(R @ v), np.linalg.norm(v)))
print("R @ R.T ≈ I ?", np.allclose(R @ R.T, np.eye(2)))

### Exercice corrigé

In [ ]:
# ---- Exercice P1 : det/inverse de M=[[3,1],[2,4]] ; rotation 90 deg ----
M2 = np.array([[3, 1], [2, 4]])
print("P1-ex : det =", round(np.linalg.det(M2),1), " ; M^-1 =\n", np.linalg.inv(M2))
assert np.isclose(np.linalg.det(M2), 10) and np.allclose(M2 @ np.linalg.inv(M2), np.eye(2))
R = np.array([[0., -1], [1, 0]])
print("P1-ex : R@(3,-4) =", R @ np.array([3., -4]), " ; R@R.T = I ?", np.allclose(R @ R.T, np.eye(2)))
assert np.allclose(R @ np.array([3., -4]), [4, 3])


## 2 · Rang, bases, sous-espaces
### Concept clé
Toute matrice `A ∈ ℝ^{m×n}` définit une application linéaire `ℝⁿ → ℝᵐ`.
Deux sous-espaces en découlent :
- **Image** (ou *rang*) : tous les vecteurs `A x`. Dimensions = `rang(A)`.
- **Noyau** (*kernel*) : tous les `x` tels que `A x = 0`.

**Théorème fondamental de l'algèbre linéaire** :
```
dim(image A) + dim(noyau A) = n   (nombre de colonnes)
```

### SVD (Singular Value Decomposition)
`A = U Σ Vᵀ` avec `U`, `V` orthogonales, `Σ` diagonale des **singulaires**
(triés décroissants). Les singulaires mesurent l'**amplitude** de A dans
chaque direction.
- Rang(A) = nombre de singulaires non nuls
- Pseudoinverse de A = `V Σ⁻¹ Uᵀ` (résout `Ax=b` au sens moindres carrés)
- Base du noyau = dernières colonnes de `V`

In [ ]:
M = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print("M de rang", np.linalg.matrix_rank(M), "(2, car col3 = col1 + col2 ?")
print("   Vrai test : col3 - 2*col2 + col1 =", M[:,2] - 2*M[:,1] + M[:,0])

U, S, Vt = np.linalg.svd(M, full_matrices=False)
print("Singulaires de M :", S)
print("Rang effectif (singulaires > tol) :", np.sum(S > 1e-9))

# Base du noyau = dernière colonne de V (ici 1 seule)
null_vec = Vt[-1]
print("Noyau ~ vecteur", null_vec, " -> M @ null_vec =", M @ null_vec)

### Exercice corrigé

In [ ]:
# ---- Exercice P2 : la famille est-elle une base de R^3 ? ----
v = np.array([[1, 2, 3], [2, 4, 6], [1, 1, 1]])
r = np.linalg.matrix_rank(v)
print(f"P2-ex : rang = {r} -> base ? {r==3}  (col2 = 2*col1)")
assert r < 3


## 3 · Valeurs propres
`Av = λv` : le vecteur `v` est simplement **étiré** par le facteur `λ`.
Pour une matrice symétrique : valeurs propres **réelles**, vecteurs
**orthogonaux** → on utilise `np.linalg.eigh` (plus stable).

**Interprétation ML :**
- Matrice de covariance `C = XᵀX/n` → ses valeurs propres = **variances**
  sur les directions principales.
- Les vecteurs propres = les **axes principaux** = PCA.
- En graphes : la valeur propre max du laplacien renseigne sur la connectivité.

In [ ]:
P = np.array([[2, 1], [1, 2]])   # symétrique
eigs, eigvecs = np.linalg.eigh(P)
print("Valeurs propres :", eigs)
print("Vecteurs propres (colonnes) :\n", eigvecs)
print("Vérif P@v_i = λ_i*v_i :", P @ eigvecs[:, 0],
      "=", eigs[0], "* ", eigvecs[:, 0])
assert np.allclose(P @ eigvecs, eigvecs * eigs)
print("\nSomme des valeurs propres =", eigs.sum(), "(= trace)")
print("Produit des valeurs propres =", eigs.prod(), "(= det)")

### Exercice corrigé

In [ ]:
# ---- Exercice P3 : valeurs propres de [[4,1],[1,3]] ----
Q = np.array([[4, 1], [1, 3]])
e, _ = np.linalg.eigh(Q)
print(f"P3-ex : valeurs propres = {e}")
assert np.isclose(e.sum(), 7) and np.isclose(e[0]*e[1], 11)


## 4 · SVD — la décomposition universelle
`A = U Σ Vᵀ`. C'est **plus général** que les valeurs propres (travaille
sur toute matrice, pas seulement carrée). Applications phares :
- **Compression** : garder seulement les `k` premiers singulaires →
  approximation `A ≈ Uₖ Σₖ Vₖᵀ` (théorème d'Eckart-Young : meilleure
  approximation de rang `k`).
- **Recommandation** : `X ≈ Uₖ Σₖ Vₖᵀ` sur une matrice utilisateur-article.
- **PCA** : SVD de la matrice centrée = PCA.

In [ ]:
X = np.array([[9, 3, 4],
              [2, 6, 7]])
U, S, Vt = np.linalg.svd(X, full_matrices=False)
print("Singulaires :", S)
print("Reconstruction U@diag(S)@Vt =\n", U @ np.diag(S) @ Vt)

# Approximation de rang 1 (théorème d'Eckart-Young)
X_r1 = U[:, :1] @ np.diag(S[:1]) @ Vt[:1, :]
print("\nApproximation rang 1 :\n", X_r1)
print("Erreur (Frobenius) :||X - X_r1|| =", np.linalg.norm(X - X_r1))

### Exercice corrigé

In [ ]:
# ---- Exercice P4 : erreur de l'approximation SVD de rang 1 ----
X = np.array([[9, 3, 4], [2, 6, 7]])
U, S, Vt = np.linalg.svd(X, full_matrices=False)
Xr1 = U[:, :1] @ np.diag(S[:1]) @ Vt[:1]
err = np.linalg.norm(X - Xr1)
print(f"P4-ex : ||X - X_rang1|| = {err:.4f}  (= S[1] = {S[1]:.4f}, Eckart-Young)")
assert np.isclose(err, S[1])


## 5 · PCA, eigendecomposition & Cholesky *(MML ch. 4.3-4.4)*
Données centrées → SVD → les colonnes de `V` (lignes de `Vᵀ`) sont les
composantes principales, et `Sᵢ²` = variance expliquée.

Le ratio `s₁² / Σ sᵢ²` donne la part de variance captée par PC1.
C'est la même chose que sklearn.decomposition.PCA, mais sans l'outil.

In [ ]:
rng = np.random.default_rng(0)
# nuage 2D allongé (corrélation forte entre dimensions)
pts = rng.normal(0, 1, (100, 2)) * np.array([3, 1])
Xc = pts - pts.mean(axis=0)
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
pc1 = Vt[0]
var_expl = (S[0] ** 2) / np.sum(S ** 2)
print("Direction PC1 =", pc1)
print(f"Part de variance expliquée par PC1 = {var_expl:.3f}")

# Vérif avec sklearn
from sklearn.decomposition import PCA
pca = PCA().fit(Xc)
print("sklearn PC1 =", pca.components_[0], "  ratio =", pca.explained_variance_ratio_[0])
assert np.allclose(np.abs(pca.components_[0]), np.abs(pc1))
assert np.isclose(pca.explained_variance_ratio_[0], var_expl)

### Eigendecomposition & diagonalisation *(MML ch. 4.4)*

Si `A` possède `n` vecteurs propres linéairement indépendants, alors
`A = P Λ P⁻¹` : c'est **diagonaliser** A. Conséquences majeures :
- les puissances `Aⁿ = P Λⁿ P⁻¹` se calculent en une multiplication diagonale
- les exponentielles de matrices `eᴬ = P e^Λ P⁻¹` (équations différentielles, noyaux)
- pour A **symétrique**, P est orthogonale : `P⁻¹ = Pᵀ` (théorème spectral, T01 §5)

**Limite** : une matrice peut ne pas être diagonalisable (valeur propre
multiple mais vecteur propre unique) — c'est le cas des matrices
**nilpotentes** non nulles.

In [ ]:
A = np.array([[4., -2.], [1., 1.]])
eigs, P = np.linalg.eig(A)
Lam = np.diag(eigs)
print("Valeurs propres :", eigs)
print("Reconstruction P @ Λ @ P⁻¹ =\n", P @ Lam @ np.linalg.inv(P))
assert np.allclose(P @ Lam @ np.linalg.inv(P), A)

# Puissance de matrice : A^15 par diagonalisation (vs itératif)
n = 15
An_ref = np.linalg.matrix_power(A, n)
An_fast = P @ np.diag(eigs ** n) @ np.linalg.inv(P)
print(f"A^{n} identique par diagonalisation :", np.allclose(An_ref, An_fast))
assert np.allclose(An_ref, An_fast)

# Contre-exemple : B = [[0,1],[0,0]] (nilpotente, B² = 0 mais B ≠ 0)
B = np.array([[0., 1.], [0., 0.]])
eb, Pb = np.linalg.eig(B)
print("\nB nilpotente : valeur propre double =", eb,
      ", rang(P) =", np.linalg.matrix_rank(Pb), "-> NON diagonalisable")
assert np.linalg.matrix_rank(Pb) < 2

### Décomposition de Cholesky *(MML ch. 4.3)*

Pour toute matrice **symétrique définie positive (SPD)** `C`, il existe
une triangulaire inférieure `L` telle que `C = L Lᵀ`.

**Pourquoi on l'utilise :**
- résoudre `Cx = b` avec une matrice SPD : 2× moins coûteux qu'un LU
- **échantillonner un Gaussien corrélé** : si `z ~ N(0, I)` alors
  `x = L z` a covariance `E[xxᵀ] = L Lᵀ = C`
- base de l'algorithme de Kalman, MCMC, et préconditionnement

In [ ]:
C = np.array([[4., 2.], [2., 3.]])
L = np.linalg.cholesky(C)
print("L =\n", L)
print("L @ L.T = C ?", np.allclose(L @ L.T, C))
assert np.allclose(L @ L.T, C)

# Échantillonner N(0, C) : x = L z
rng = np.random.default_rng(0)
z = rng.standard_normal((50000, 2))
x = z @ L.T          # Cov(x) = L Lᵀ = C
cov_est = np.cov(x.T)
print("Covariance estimée ≈ C :", np.allclose(cov_est, C, atol=0.1))
assert np.allclose(cov_est, C, atol=0.1)

### Exercice corrigé

In [ ]:
# ---- Exercice P5 : Cholesky de [[5,1],[1,2]] + valeurs propres de [[2,1],[1,2]] ----
A5 = np.array([[5.0, 1], [1, 2]])
L5 = np.linalg.cholesky(A5)
print("P5-ex : L =", L5)
assert np.allclose(L5 @ L5.T, A5) and np.isclose(L5[0,0], np.sqrt(5)) and np.isclose(L5[1,1], np.sqrt(1.8))
e5, _ = np.linalg.eigh(np.array([[2.0, 1], [1, 2]]))
print(f"P5-ex2 : eigh [[2,1],[1,2]] = {e5}  (tr = 4, det = 3)")
assert np.isclose(e5[0], 1) and np.isclose(e5[1], 3)


## 6 · PCA — perspective variable latente *(MML ch. 10.7)*
Le PCA peut se lire comme un **modèle génératif** à `k` facteurs latents :
```
x = W h + ε ,   h ~ N(0, Iₖ) ,   ε ~ N(0, σ² I)
```
- `h` : facteurs latents inobservés (dimension `k` < `n`)
- `W` : projection vers l'espace observé
- L'estimation par **maximum de vraisemblance** de `(W, σ²)` revient
  (à une rotation près) aux composantes principales du PCA.

C'est le pont vers le **factor analysis** et les GMM (T07 / MML ch. 11).

In [ ]:
rng = np.random.default_rng(42)
W_true = np.array([[3.0, 0.5], [0.2, 1.0], [2.5, -0.3]])  # 3 obs, 2 latents
m = 500
h = rng.normal(0, 1, (m, 2))
noise_sd = 0.3
X = h @ W_true.T + noise_sd * rng.normal(0, 1, (m, 3))
Xc = X - X.mean(axis=0)

pca = PCA(n_components=2).fit(Xc)
print(f"Variance expliquée par 2 facteurs : {pca.explained_variance_ratio_.sum()*100:.1f}%")
scores = pca.transform(Xc)
rec = pca.inverse_transform(scores) + X.mean(axis=0)
err = np.linalg.norm(X - rec) / np.linalg.norm(Xc)
print(f"Erreur de reconstruction relative = {err:.4f} (petite -> les 2 latents suffisent)")
assert pca.explained_variance_ratio_.sum() > 0.95
assert err < 0.10   # bruit résiduel ≈ σ_noise / amplitude

### Exercice corrigé

In [ ]:
# ---- Exercice P6 : eigendecomposition de corr = [[1,0.9],[0.9,1]] ----
e6, V6 = np.linalg.eigh(np.array([[1.0, 0.9], [0.9, 1]]))
print(f"P6-ex : valeurs propres = {e6} ; premier axe = {np.round(V6[:,0],3)}")
assert np.isclose(e6[0], 0.1) and np.isclose(e6[1], 1.9)
assert np.allclose(np.abs(V6[:,1]), [1/np.sqrt(2)]*2)


## 7 · Synthèse — exercices intégrés

### Exercice corrigé

In [ ]:
# ---- Exercice P7 : projection orthogonale de u=(3,4) sur (1,0) et (1,1) ----
u = np.array([3.0, 4.0])
for vv in [np.array([1.0, 0.0]), np.array([1.0, 1.0])]:
    proj = (u @ vv) / (vv @ vv) * vv
    print(f"P7-ex : proj de u sur {vv} = {proj}")
assert np.allclose(proj, [3.5, 3.5])
print("\n✅ T01 termine — 7 points, exercices integres.")
